# Эксперименты 4–5: словоформа, лексема и смысл

Описание методики и результатов — в `README.md` репозитория.

## 1. Подготовить среду

In [ ]:
%pip -q install "transformers==4.48.3" "accelerate>=1.2.0" "bitsandbytes>=0.45.0" "wordfreq>=3.1.1" pymorphy3 pymorphy3-dicts-ru pandas matplotlib tqdm

## 2. Настройки

In [ ]:
import os
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from wordfreq import top_n_list, zipf_frequency
import pymorphy3
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

SEED = 42
MODEL_NAME = "IlyaGusev/saiga_llama3_8b"
TOP_N_WORDS = 100_000
N_LEMMAS = 250
MIN_ZIPF = 2.0
MIN_FORMS = 5
MAX_PAIRS = 5_000
BATCH_SIZE = 16
USE_4BIT = True
RESULT_DIR = "/content/saiga_morphology_results"
os.makedirs(RESULT_DIR, exist_ok=True)

rng = np.random.default_rng(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Нужен GPU: в Colab выберите Runtime → Change runtime type → GPU.")
print("GPU:", torch.cuda.get_device_name(0))

## 3. Загрузить модель

In [ ]:
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
PAD_ID, BOS_ID = tokenizer.pad_token_id, tokenizer.bos_token_id

load_kwargs = {"device_map": "auto", "torch_dtype": COMPUTE_DTYPE, "low_cpu_mem_usage": True}
if USE_4BIT:
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
model.eval()
DEVICE = model.get_input_embeddings().weight.device
N_STATES = model.config.num_hidden_layers + 1
HIDDEN_SIZE = model.config.hidden_size
print("Слоёв (с embedding-слоем):", N_STATES)

## 4. Материал 1: существительные и их формы (словоизменение)

In [ ]:
morph = pymorphy3.MorphAnalyzer()
CASES = ["nomn", "gent", "datv", "accs", "ablt", "loct"]
NUMBERS = ["sing", "plur"]
SKIP_TAGS = {"Name", "Surn", "Patr", "Geox", "Orgn", "Trad", "Abbr", "Init", "Fixd", "Pltm", "Sgtm"}


def paradigm(word):
    p = morph.parse(word)[0]
    if p.tag.POS != "NOUN" or p.normal_form != word or any(t in p.tag for t in SKIP_TAGS):
        return None
    if p.tag.case != "nomn" or p.tag.number != "sing":
        return None
    slots = {}
    for f in p.lexeme:
        if f.tag.case in CASES and f.tag.number in NUMBERS:
            slots.setdefault((f.tag.case, f.tag.number), f.word)
    counts = defaultdict(int)
    for w in slots.values():
        counts[w] += 1
    return {slot: w for slot, w in slots.items() if counts[w] == 1}


lemma_forms = {}
for raw in tqdm(top_n_list("ru", TOP_N_WORDS), desc="Поиск существительных"):
    w = raw.strip().lower()
    if not re.fullmatch(r"[а-я]+", w) or len(w) < 3 or w in lemma_forms:
        continue
    forms = paradigm(w)
    if forms:
        forms = {slot: f for slot, f in forms.items() if zipf_frequency(f, "ru") >= MIN_ZIPF}
        if len(forms) >= MIN_FORMS:
            lemma_forms[w] = forms
    if len(lemma_forms) >= N_LEMMAS * 2:
        break

owner = defaultdict(set)
for lemma, forms in lemma_forms.items():
    for f in forms.values():
        owner[f].add(lemma)
lemma_forms = {lemma: {s: f for s, f in forms.items() if len(owner[f]) == 1}
               for lemma, forms in lemma_forms.items()}
lemma_forms = {l: f for l, f in lemma_forms.items() if len(f) >= MIN_FORMS}
lemma_forms = dict(list(lemma_forms.items())[:N_LEMMAS])

noun_rows = []
for lemma, forms in lemma_forms.items():
    for (case, number), form in forms.items():
        ids = tokenizer.encode(" " + form, add_special_tokens=False)
        noun_rows.append({"lemma": lemma, "form": form, "case": case, "number": number,
                          "slot": f"{case}_{number}", "token_ids": ids})
nouns = pd.DataFrame(noun_rows)
nouns["n_tokens"] = nouns["token_ids"].str.len()
nouns["first_tok"] = nouns["token_ids"].str[0]
nouns["last_tok"] = nouns["token_ids"].str[-1]
nouns["pieces"] = nouns["token_ids"].apply(lambda ids: "|".join(tokenizer.decode([i]).strip() for i in ids))

print(f"Слов: {nouns['lemma'].nunique()}, форм: {len(nouns)}, "
      f"в среднем {len(nouns) / nouns['lemma'].nunique():.1f} формы на слово")
print("Сколько форм из 1/2/3+ токенов:", nouns["n_tokens"].clip(upper=3).value_counts().sort_index().to_dict())
example = nouns[nouns["lemma"] == nouns["lemma"].iloc[0]]
print(f"Пример — «{example['lemma'].iloc[0]}»:")
display(example[["slot", "form", "pieces"]])

## 5. Материал 2 и 3: словообразование и супплетивные пары

In [ ]:
DERIVATION_PAIRS = {
    "уменьшительное": [("стол", "столик"), ("дом", "домик"), ("кот", "котик"), ("нос", "носик"),
                       ("сад", "садик"), ("мост", "мостик"), ("ключ", "ключик"), ("мяч", "мячик"),
                       ("лес", "лесок"), ("город", "городок"), ("книга", "книжка"), ("река", "речка"),
                       ("птица", "птичка"), ("сын", "сынок"), ("друг", "дружок"), ("нога", "ножка"),
                       ("дождь", "дождик"), ("голова", "головка")],
    "деятель": [("учить", "учитель"), ("писать", "писатель"), ("читать", "читатель"),
                ("строить", "строитель"), ("водить", "водитель"), ("слушать", "слушатель"),
                ("хранить", "хранитель"), ("спасать", "спасатель"), ("руководить", "руководитель"),
                ("преподавать", "преподаватель"), ("продавать", "продавец"), ("петь", "певец"),
                ("бегать", "бегун"), ("играть", "игрок")],
    "прилагательное от сущ.": [("лес", "лесной"), ("город", "городской"), ("море", "морской"),
                               ("книга", "книжный"), ("вода", "водный"), ("школа", "школьный"),
                               ("зима", "зимний"), ("мир", "мировой"), ("дом", "домашний"),
                               ("музыка", "музыкальный"), ("история", "исторический"),
                               ("наука", "научный"), ("война", "военный"), ("солнце", "солнечный"),
                               ("ночь", "ночной")],
    "отвлечённое от прил.": [("добрый", "доброта"), ("красивый", "красота"), ("быстрый", "быстрота"),
                             ("слабый", "слабость"), ("молодой", "молодость"), ("старый", "старость"),
                             ("глупый", "глупость"), ("честный", "честность"), ("свежий", "свежесть"),
                             ("высокий", "высота"), ("широкий", "ширина"), ("глубокий", "глубина"),
                             ("тихий", "тишина")],
    "приставочный глагол": [("писать", "переписать"), ("читать", "прочитать"), ("делать", "сделать"),
                            ("ходить", "выходить"), ("бежать", "убежать"), ("строить", "перестроить"),
                            ("думать", "придумать"), ("нести", "принести"), ("ехать", "уехать"),
                            ("играть", "выиграть")],
}

SUPPLETION_PAIRS = [("человек", "люди"), ("ребенок", "дети"), ("идти", "шел"), ("хороший", "лучше"),
                    ("плохой", "хуже"), ("хорошо", "лучше"), ("плохо", "хуже"), ("много", "больше"),
                    ("мало", "меньше"), ("маленький", "меньше"), ("я", "меня"), ("ты", "тебя"),
                    ("мы", "нас"), ("вы", "вас"), ("он", "его"), ("она", "ее"), ("год", "лет"),
                    ("брать", "взять"), ("говорить", "сказать"), ("класть", "положить"),
                    ("ловить", "поймать"), ("садиться", "сесть"), ("ложиться", "лечь")]

enc = lambda w: tokenizer.encode(" " + w, add_special_tokens=False)
deriv = pd.DataFrame([{"type": t, "base": b, "derived": d} for t, pairs in DERIVATION_PAIRS.items()
                      for b, d in pairs])
supp = pd.DataFrame([{"a": a, "b": b, "shared_tokens": bool(set(enc(a)) & set(enc(b)))}
                     for a, b in SUPPLETION_PAIRS])
dropped = supp[supp["shared_tokens"]]
supp = supp[~supp["shared_tokens"]].reset_index(drop=True)
print(f"Словообразовательных пар: {len(deriv)}")
print(f"Супплетивных пар без общих токенов: {len(supp)}"
      + (f" (убраны из-за общих токенов: {', '.join(dropped['a'] + '–' + dropped['b'])})" if len(dropped) else ""))

## 6. Векторы последнего токена на всех слоях

In [ ]:
all_words = list(dict.fromkeys(list(nouns["form"]) + list(deriv["base"]) + list(deriv["derived"])
                               + list(supp["a"]) + list(supp["b"])))
word_idx = {w: i for i, w in enumerate(all_words)}
word_ids = [enc(w) for w in all_words]

feats = np.empty((N_STATES, len(all_words), HIDDEN_SIZE), dtype=np.float16)
for start in tqdm(range(0, len(all_words), BATCH_SIZE), desc="Векторы"):
    batch = word_ids[start:start + BATCH_SIZE]
    rows = [[BOS_ID] + ids for ids in batch]
    max_len = max(len(r) for r in rows)
    input_ids = torch.full((len(rows), max_len), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(rows), max_len), dtype=torch.long)
    for i, r in enumerate(rows):
        input_ids[i, :len(r)] = torch.tensor(r)
        mask[i, :len(r)] = 1
    with torch.inference_mode():
        out = model(input_ids=input_ids.to(DEVICE), attention_mask=mask.to(DEVICE),
                    output_hidden_states=True, use_cache=False)
    r_t = torch.arange(len(rows), device=DEVICE)
    last = torch.tensor([len(r) - 1 for r in rows], device=DEVICE)
    for layer, h in enumerate(out.hidden_states):
        feats[layer, start:start + len(rows)] = h[r_t, last].to(torch.float16).cpu().numpy()
    del out


def normalized_layer(layer):
    X = feats[layer].astype(np.float32)
    X = X - X.mean(axis=0, keepdims=True)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)


print("Векторы:", feats.shape)

## 7. Измерение 1: похожесть пар с контролями

In [ ]:
N = len(nouns)
lemmas = nouns["lemma"].values
first = nouns["first_tok"].values
lastt = nouns["last_tok"].values
slots = nouns["slot"].values
tok_sets = [set(ids) for ids in nouns["token_ids"]]
noun_idx = np.array([word_idx[f] for f in nouns["form"]])


def sample(pairs):
    pairs = list(pairs)
    if len(pairs) > MAX_PAIRS:
        pairs = [pairs[i] for i in rng.choice(len(pairs), MAX_PAIRS, replace=False)]
    return np.array(pairs, dtype=np.int64).reshape(-1, 2)


same_lemma = [(i, j) for l in np.unique(lemmas) for k, i in enumerate(np.where(lemmas == l)[0])
              for j in np.where(lemmas == l)[0][k + 1:]]
PAIRS = {}
PAIRS["А1. Формы одного слова, общая первая часть"] = sample(
    (i, j) for i, j in same_lemma if first[i] == first[j] and lastt[i] != lastt[j])
PAIRS["А2. Формы одного слова, нет общих кусков"] = sample(
    (i, j) for i, j in same_lemma if not (tok_sets[i] & tok_sets[j]))

by_first = defaultdict(list)
for i in range(N):
    by_first[first[i]].append(i)
PAIRS["Б. Разные слова, общая первая часть"] = sample(
    (i, j) for group in by_first.values() for k, i in enumerate(group) for j in group[k + 1:]
    if lemmas[i] != lemmas[j] and lastt[i] != lastt[j])


def random_pairs(condition, n=MAX_PAIRS, tries=50 * MAX_PAIRS):
    out, a, b = set(), rng.integers(0, N, tries), rng.integers(0, N, tries)
    for i, j in zip(a, b):
        if i != j and lemmas[i] != lemmas[j] and condition(i, j):
            out.add((min(i, j), max(i, j)))
            if len(out) >= n:
                break
    return np.array(sorted(out), dtype=np.int64).reshape(-1, 2)


PAIRS["В. Разные слова, нет общих кусков"] = random_pairs(lambda i, j: not (tok_sets[i] & tok_sets[j]))
PAIRS["Г. Разные слова, та же грамм. форма"] = random_pairs(lambda i, j: slots[i] == slots[j])

for name, p in PAIRS.items():
    print(f"{name}: {len(p)} пар")
    if len(p) < 30:
        print("   Внимание: пар мало, оценка будет неточной")

sim_rows = []
for layer in tqdm(range(N_STATES), desc="Похожесть по слоям"):
    X = normalized_layer(layer)
    for name, p in PAIRS.items():
        if len(p) == 0:
            continue
        s = (X[noun_idx[p[:, 0]]] * X[noun_idx[p[:, 1]]]).sum(axis=1)
        sim_rows.append({"layer": layer, "pair_type": name, "mean": s.mean(),
                         "se": s.std(ddof=1) / np.sqrt(len(s)) if len(s) > 1 else np.nan, "n": len(s)})
sim_df = pd.DataFrame(sim_rows)
sim_df.to_csv(os.path.join(RESULT_DIR, "pair_similarity_by_layer.csv"), index=False)


def effect(a, b):
    layers = pd.Index(range(N_STATES), name="layer")
    A = sim_df[sim_df.pair_type == a].set_index("layer").reindex(layers)
    B = sim_df[sim_df.pair_type == b].set_index("layer").reindex(layers)
    diff = A["mean"] - B["mean"]
    half = 1.96 * np.sqrt(A["se"] ** 2 + B["se"] ** 2)
    return pd.DataFrame({"diff": diff, "low": diff - half, "high": diff + half})


eff_first = effect("А1. Формы одного слова, общая первая часть", "Б. Разные слова, общая первая часть")
eff_none = effect("А2. Формы одного слова, нет общих кусков", "В. Разные слова, нет общих кусков")
print("Готово.")

## 8. Измерение 2: кто ближайший сосед — то же слово или та же форма?

In [ ]:
nn_tasks = []
for i in range(N):
    own = np.where((lemmas == lemmas[i]) & (np.arange(N) != i))[0]
    others = np.where((slots == slots[i]) & (lemmas != lemmas[i]))[0]
    k = min(len(own), len(others))
    if k == 0:
        continue
    nn_tasks.append((i, rng.choice(own, k, replace=False), rng.choice(others, k, replace=False)))


def wilson(p, n, z=1.96):
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return c - h, c + h


nn_rows = []
for layer in range(N_STATES):
    X = normalized_layer(layer)[noun_idx]
    hits = [(X[own] @ X[i]).max() > (X[oth] @ X[i]).max() for i, own, oth in nn_tasks]
    p = float(np.mean(hits))
    lo, hi = wilson(p, len(hits))
    nn_rows.append({"layer": layer, "lexeme_share": p, "low": lo, "high": hi, "n": len(hits)})
nn_df = pd.DataFrame(nn_rows)
nn_df.to_csv(os.path.join(RESULT_DIR, "nearest_neighbor_lexeme_vs_form.csv"), index=False)
best = nn_df.loc[nn_df["lexeme_share"].idxmax()]
print(f"Задач: {len(nn_tasks)}. Максимум «то же слово»: {best['lexeme_share']:.1%} на слое {int(best['layer'])}; "
      f"минимум: {nn_df['lexeme_share'].min():.1%} на слое {int(nn_df.loc[nn_df['lexeme_share'].idxmin(), 'layer'])}")

## 9. Измерение 3: словообразование и супплетивные пары

In [ ]:
all_same_lemma = sample(same_lemma)
extra_rows = []
for layer in range(N_STATES):
    X = normalized_layer(layer)
    groups = {
        "Формы одного слова (все пары)": (noun_idx[all_same_lemma[:, 0]], noun_idx[all_same_lemma[:, 1]]),
        "Словообразование": (deriv["base"].map(word_idx).values, deriv["derived"].map(word_idx).values),
        "Супплетивные пары": (supp["a"].map(word_idx).values, supp["b"].map(word_idx).values),
    }
    for t, g in deriv.groupby("type"):
        groups[f"Словообразование: {t}"] = (g["base"].map(word_idx).values, g["derived"].map(word_idx).values)
    for name, (a, b) in groups.items():
        s = (X[a] * X[b]).sum(axis=1)
        extra_rows.append({"layer": layer, "pair_type": name, "mean": s.mean(),
                           "se": s.std(ddof=1) / np.sqrt(len(s)), "n": len(s)})
extra_df = pd.DataFrame(extra_rows)
both_df = pd.concat([sim_df, extra_df], ignore_index=True)
both_df.to_csv(os.path.join(RESULT_DIR, "all_similarity_by_layer.csv"), index=False)
print("Готово.")

## 10. Результаты

In [ ]:
COL = {"А1. Формы одного слова, общая первая часть": "#2a78d6",
       "Б. Разные слова, общая первая часть": "#9cc2ef",
       "А2. Формы одного слова, нет общих кусков": "#1baf7a",
       "В. Разные слова, нет общих кусков": "#9adfc4",
       "Г. Разные слова, та же грамм. форма": "#eb6834",
       "Формы одного слова (все пары)": "#2a78d6",
       "Словообразование": "#4a3aa7",
       "Супплетивные пары": "#e87ba4"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10})


def line(ax, name, label=None, ls="-"):
    d = both_df[both_df.pair_type == name].sort_values("layer")
    if d.empty:
        return
    ax.plot(d["layer"], d["mean"], color=COL[name], lw=2, ls=ls, label=label or name)
    ax.fill_between(d["layer"], d["mean"] - 1.96 * d["se"], d["mean"] + 1.96 * d["se"],
                    color=COL[name], alpha=0.15, lw=0)


fig, axes = plt.subplots(2, 2, figsize=(15, 10))
ax = axes[0, 0]
for name in ["А1. Формы одного слова, общая первая часть", "Б. Разные слова, общая первая часть",
             "А2. Формы одного слова, нет общих кусков", "В. Разные слова, нет общих кусков",
             "Г. Разные слова, та же грамм. форма"]:
    line(ax, name, ls="--" if name[0] in "БВ" else "-")
ax.set_title("1. Похожесть пар (пунктир — контроли)", loc="left")
ax.set_ylabel("Косинусная похожесть (центрированная)")

ax = axes[0, 1]
ax.plot(nn_df["layer"], nn_df["lexeme_share"], color="#2a78d6", lw=2, marker="o", ms=3,
        label="Ближайший сосед — форма того же слова")
ax.fill_between(nn_df["layer"], nn_df["low"], nn_df["high"], color="#2a78d6", alpha=0.15, lw=0)
ax.axhline(0.5, ls="--", lw=1, color="#52514e", label="50%: нет предпочтения")
ax.set_ylim(0, 1)
ax.set_title("2. Лексема (выше 50%) или грамматическая форма (ниже 50%)?", loc="left")
ax.set_ylabel("Доля")

ax = axes[1, 0]
line(ax, "Формы одного слова (все пары)", "Словоизменение: формы одного слова")
line(ax, "Словообразование", "Словообразование: исходное → производное")
line(ax, "Б. Разные слова, общая первая часть", "Контроль: разные слова, общая первая часть", ls="--")
line(ax, "В. Разные слова, нет общих кусков", "Планка: разные слова, нет общих кусков", ls="--")
ax.set_title("3. Словоизменение и словообразование", loc="left")
ax.set_ylabel("Косинусная похожесть (центрированная)")

ax = axes[1, 1]
line(ax, "Супплетивные пары", f"Супплетивные пары ({len(supp)})")
line(ax, "В. Разные слова, нет общих кусков", "Разные слова, нет общих кусков", ls="--")
ax.set_title("4. Пары без общих кусков: «человек — люди», «я — меня»", loc="left")

for ax in axes.flat:
    ax.set_xlabel("Слой (0 = embedding)")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=8.5)
fig.suptitle("Saiga Llama 3 8B: что собирается — словоформа или лексема?", x=0.01, ha="left", fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(RESULT_DIR, "morphology_results.png"), dpi=180, bbox_inches="tight")
plt.show()

## 11. Итог по критерию

In [ ]:
crit = pd.DataFrame({
    "layer": range(N_STATES),
    "А1 > Б": eff_first["low"].values > 0,
    "А2 > В": eff_none["low"].values > 0,
    "сосед — то же слово > 50%": nn_df["low"].values > 0.5,
    "сосед — та же форма > 50%": nn_df["high"].values < 0.5,
})
crit["лексема"] = crit["А1 > Б"] & crit["А2 > В"] & crit["сосед — то же слово > 50%"]
show = crit.copy()
for c in show.columns[1:]:
    show[c] = show[c].map({True: "да", False: ""})
show["А1 − Б"] = eff_first["diff"].round(3).values
show["А2 − В"] = eff_none["diff"].round(3).values
show["доля «то же слово»"] = nn_df["lexeme_share"].round(3).values
display(show)
crit.to_csv(os.path.join(RESULT_DIR, "criterion_by_layer.csv"), index=False)


def ranges(layers):
    out, start = [], None
    for k, l in enumerate(layers):
        if start is None:
            start = l
        if k == len(layers) - 1 or layers[k + 1] != l + 1:
            out.append(f"{start}–{l}" if l != start else f"{l}")
            start = None
    return ", ".join(out) if out else "нет"


lex_layers = crit.loc[crit["лексема"], "layer"].tolist()
form_layers = crit.loc[crit["сосед — та же форма > 50%"], "layer"].tolist()
peak = int(nn_df["lexeme_share"].idxmax())


def mean_at(name, layer):
    v = both_df[(both_df.pair_type == name) & (both_df.layer == layer)]["mean"]
    return v.item() if len(v) == 1 else float("nan")


print("ВЫВОД")
print(f"• Все три условия «модель собирает лексему» выполнены на слоях: {ranges(lex_layers)}.")
print(f"• Ближайший сосед — скорее та же грамматическая форма, чем то же слово, на слоях: {ranges(form_layers)}.")
print(f"• Сильнее всего группировка по лексеме на слое {peak}: в {nn_df.loc[peak, 'lexeme_share']:.0%} случаев "
      "ближайший сосед — форма того же слова.")
print(f"• На слое {peak}: словоизменение {mean_at('Формы одного слова (все пары)', peak):.3f}, "
      f"словообразование {mean_at('Словообразование', peak):.3f}, "
      f"контроль с общей первой частью {mean_at('Б. Разные слова, общая первая часть', peak):.3f}, "
      f"разные слова без общих кусков {mean_at('В. Разные слова, нет общих кусков', peak):.3f}.")
print(f"• Супплетивные пары на слое {peak}: {mean_at('Супплетивные пары', peak):.3f} "
      f"против {mean_at('В. Разные слова, нет общих кусков', peak):.3f} у случайных пар без общих кусков.")

print("\nСловообразование по типам на слое", peak)
display(extra_df[(extra_df.layer == peak) & extra_df.pair_type.str.startswith("Словообразование:")]
        [["pair_type", "mean", "n"]].round(3).reset_index(drop=True))

## 12. Примеры: ближайшие соседи

In [ ]:
def neighbours(word, layer, k=5):
    X = normalized_layer(layer)
    sims = X @ X[word_idx[word]]
    order = [i for i in np.argsort(-sims) if all_words[i] != word][:k]
    return ", ".join(all_words[i] for i in order)


examples = list(nouns.sort_values("n_tokens", ascending=False).drop_duplicates("lemma")["form"].head(4)) \
    + list(nouns[nouns.n_tokens == 1]["form"].head(2))
rows = [{"слово": w, **{f"слой {L}": neighbours(w, L) for L in [0, peak, N_STATES - 1]}} for w in examples]
display(pd.DataFrame(rows))

## 13. Смысловой контроль: то же слово или тот же смысл?

### Пары синонимов и слов одной смысловой группы

In [ ]:
for _name in ["model", "tokenizer", "morph", "feats", "both_df", "enc"]:
    assert _name in globals(), f"Не найдено {_name}: сначала выполните весь ноутбук по морфологии"

SYNONYMS = [("врач", "доктор"), ("автомобиль", "машина"), ("страна", "государство"), ("бой", "сражение"),
            ("задача", "задание"), ("беседа", "разговор"), ("мысль", "идея"), ("лошадь", "конь"),
            ("доход", "прибыль"), ("друг", "приятель"), ("ошибка", "промах"), ("работа", "труд"),
            ("грусть", "печаль"), ("дом", "жилище"), ("помощь", "поддержка")]
RELATED = [("журнал", "газета"), ("стол", "стул"), ("город", "село"), ("мама", "папа"), ("кошка", "собака"),
           ("брат", "сестра"), ("чай", "сок"), ("утро", "вечер"), ("зима", "лето"), ("нож", "вилка"),
           ("река", "озеро"), ("рубашка", "куртка"), ("дерево", "куст"), ("яблоко", "груша"),
           ("школа", "университет"), ("окно", "дверь"), ("король", "королева"), ("учитель", "ученик"),
           ("поезд", "автобус"), ("солнце", "луна")]


def to_slot(word, case, number):
    p = morph.parse(word)[0]
    if p.tag.POS != "NOUN":
        return None
    f = p.inflect({case, number})
    return f.word if f is not None else None


sem_rows = []
for kind, pairs in [("синонимы", SYNONYMS), ("одна смысловая группа", RELATED)]:
    for a, b in pairs:
        sem_rows.append({"kind": kind, "form": "начальная форма", "a": a, "b": b})
        fa, fb = to_slot(a, "ablt", "plur"), to_slot(b, "ablt", "plur")
        if fa and fb and fa != fb:
            sem_rows.append({"kind": kind, "form": "одна и та же форма (тв. мн.)", "a": fa, "b": fb})
sem = pd.DataFrame(sem_rows)
display(sem.groupby(["kind", "form"]).size().rename("пар").reset_index())
display(sem[sem.form == "одна и та же форма (тв. мн.)"].head(8))

### Векторы новых слов и сходство по слоям

In [ ]:
new_words = [w for w in dict.fromkeys(list(sem["a"]) + list(sem["b"])) if w not in word_idx]
new_feats = np.empty((N_STATES, len(new_words), HIDDEN_SIZE), dtype=np.float16)
for start in tqdm(range(0, len(new_words), BATCH_SIZE), desc="Векторы новых слов"):
    batch = [enc(w) for w in new_words[start:start + BATCH_SIZE]]
    rows = [[BOS_ID] + ids for ids in batch]
    max_len = max(len(r) for r in rows)
    input_ids = torch.full((len(rows), max_len), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(rows), max_len), dtype=torch.long)
    for i, r in enumerate(rows):
        input_ids[i, :len(r)] = torch.tensor(r)
        mask[i, :len(r)] = 1
    with torch.inference_mode():
        out = model(input_ids=input_ids.to(DEVICE), attention_mask=mask.to(DEVICE),
                    output_hidden_states=True, use_cache=False)
    r_t = torch.arange(len(rows), device=DEVICE)
    last = torch.tensor([len(r) - 1 for r in rows], device=DEVICE)
    for layer, h in enumerate(out.hidden_states):
        new_feats[layer, start:start + len(rows)] = h[r_t, last].to(torch.float16).cpu().numpy()
    del out

new_idx = {w: i for i, w in enumerate(new_words)}


def vec(layer, word, mean):
    v = (feats[layer, word_idx[word]] if word in word_idx else new_feats[layer, new_idx[word]]).astype(np.float32)
    v = v - mean
    return v / (np.linalg.norm(v) + 1e-8)


sem_sim_rows = []
for layer in range(N_STATES):
    mean = feats[layer].astype(np.float32).mean(axis=0)
    for (kind, form), g in sem.groupby(["kind", "form"]):
        s = np.array([vec(layer, a, mean) @ vec(layer, b, mean) for a, b in zip(g["a"], g["b"])])
        sem_sim_rows.append({"layer": layer, "pair_type": f"Близкие по смыслу: {kind}, {form}",
                             "mean": s.mean(), "se": s.std(ddof=1) / np.sqrt(len(s)), "n": len(s)})
sem_df = pd.DataFrame(sem_sim_rows)
sem_df.to_csv(os.path.join(RESULT_DIR, "semantic_control_by_layer.csv"), index=False)
print("Готово:", len(new_words), "новых слов")

### Сравнение и вывод

In [ ]:
FORMS = "Формы одного слова (все пары)"
SAME_FORM = "Г. Разные слова, та же грамм. форма"
RANDOM = "В. Разные слова, нет общих кусков"
SYN_NOM = "Близкие по смыслу: синонимы, начальная форма"
SYN_SAME = "Близкие по смыслу: синонимы, одна и та же форма (тв. мн.)"
REL_NOM = "Близкие по смыслу: одна смысловая группа, начальная форма"
REL_SAME = "Близкие по смыслу: одна смысловая группа, одна и та же форма (тв. мн.)"
all_df = pd.concat([both_df, sem_df], ignore_index=True)


def curve(name):
    return all_df[all_df.pair_type == name].set_index("layer").reindex(range(N_STATES))


lines = [(FORMS, "#2a78d6", "-", "Формы одного слова («журнал — журналами»)"),
         (SYN_NOM, "#eb6834", "-", "Синонимы («врач — доктор»)"),
         (SYN_SAME, "#eb6834", ":", "Синонимы в одной форме («врачами — докторами»)"),
         (REL_NOM, "#1baf7a", "-", "Одна смысловая группа («журнал — газета»)"),
         (REL_SAME, "#1baf7a", ":", "Одна смысловая группа в одной форме («журналами — газетами»)"),
         (SAME_FORM, "#9aa0a6", "--", "Разные слова в одной форме, случайные («столом — домом»)"),
         (RANDOM, "#c9cbcf", "--", "Случайные слова без общих кусков")]
fig, ax = plt.subplots(figsize=(12, 6))
for name, color, ls, label in lines:
    d = curve(name)
    if d["mean"].isna().all():
        continue
    ax.plot(d.index, d["mean"], color=color, ls=ls, lw=2, label=label)
    ax.fill_between(d.index, d["mean"] - 1.96 * d["se"], d["mean"] + 1.96 * d["se"], color=color, alpha=0.12, lw=0)
ax.set_xlabel("Слой (0 = embedding)")
ax.set_ylabel("Косинусная похожесть (центрированная)")
ax.set_title("То же слово или тот же смысл?", loc="left")
ax.grid(alpha=0.25)
ax.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, "semantic_control.png"), dpi=180, bbox_inches="tight")
plt.show()


def beats(a, b):
    A, B = curve(a), curve(b)
    low = (A["mean"] - B["mean"]) - 1.96 * np.sqrt(A["se"] ** 2 + B["se"] ** 2)
    return [int(l) for l in low.index[low > 0]]


def ranges(layers):
    out, start = [], None
    for k, l in enumerate(layers):
        start = l if start is None else start
        if k == len(layers) - 1 or layers[k + 1] != l + 1:
            out.append(f"{start}–{l}" if l != start else f"{l}")
            start = None
    return ", ".join(out) if out else "нет"


summary = []
for name, _, _, label in lines:
    d = curve(name)
    if d["mean"].isna().all():
        continue
    row = {"Тип пары": label, "Пар": int(d["n"].iloc[0])}
    for L in [0, 5, 15, 25, N_STATES - 1]:
        if L < N_STATES:
            row[f"Слой {L}"] = round(d["mean"].loc[L], 3)
    summary.append(row)
display(pd.DataFrame(summary))

print("ВЫВОД")
for other, label in [(SYN_NOM, "синонимы"), (SYN_SAME, "синонимы в одной форме"),
                     (REL_NOM, "слова одной смысловой группы"), (REL_SAME, "слова одной смысловой группы в одной форме")]:
    if curve(other)["mean"].isna().all():
        continue
    print(f"• Формы одного слова достоверно ближе, чем {label}, на слоях: {ranges(beats(FORMS, other))}; "
          f"наоборот — на слоях: {ranges(beats(other, FORMS))}.")
print(f"• Слова одной смысловой группы в одной форме ближе, чем случайные слова в одной форме, на слоях: "
      f"{ranges(beats(REL_SAME, SAME_FORM))} (это и есть вклад смысла).")

## 14. Скачать результаты

In [ ]:
from google.colab import files
for name in ["morphology_results.png", "criterion_by_layer.csv", "pair_similarity_by_layer.csv",
             "nearest_neighbor_lexeme_vs_form.csv", "all_similarity_by_layer.csv"]:
    files.download(os.path.join(RESULT_DIR, name))
nouns.drop(columns=["token_ids"]).to_csv(os.path.join(RESULT_DIR, "noun_forms.csv"), index=False)
files.download(os.path.join(RESULT_DIR, "noun_forms.csv"))